In [ ]:
import gurobipy as gp
import pandas as pd
import numpy as np
from matpowercaseframes import CaseFrames
from gurobi_optimods import opf
# from gurobi_optimods.opf import converters, grbformulator, violations


In [ ]:
opftype = "dc"
useef = False
usejabr = False
default_solver_params = {"MIPGap": 1e-4, "OptimalityTol": 1e-4}
with create_env(params=default_solver_params) as env:
	return _solve_opf_model_internal(
		env,
		case,
		opftype=opftype,
		useef=useef,
		usejabr=usejabr,
		branchswitching=branch_switching,
		usemipstart=use_mip_start,
		minactivebranches=min_active_branches,
		polar=False,
		ivtype="aggressive",
		useactivelossineqs=False,
	)

In [122]:
use_mip_start=False
usemipstart=use_mip_start
min_active_branches=0.9
minactivebranches=min_active_branches
branch_switching=False
branchswitching=branch_switching
useactivelossineqs=False
ivtype="aggressive"
polar=False,
opftype = "dc"
useef = False
usejabr = False
default_solver_params = {"MIPGap": 1e-4, "OptimalityTol": 1e-4}
settings = opf.converters.build_internal_settings(
	opftype,
	polar,
	useef,
	usejabr,
	ivtype,
	branchswitching,
	usemipstart,
	minactivebranches,
	useactivelossineqs,
)

In [123]:
case_path = './old/DCOPF-main/power_system_test_cases/pglib_opf_case300_ieee.mat'
case = opf.read_case_matpower(case_path)
alldata = opf.converters.convert_case_to_internal_format(case)
alldata.update(settings)

In [102]:
# Gurobipy implementation
case_path = './pglib-opf-21.07/pglib_opf_case300_ieee.m'
ref = CaseFrames(case_path)
# print(f"{ref.bus.PD.sum()/1000:.2f}") # GW
p_max = ref.gen["PMAX"].values    # or ref.gen.PMAX if it's an attribute
p_inf = max_gen = p_max.max()
p_one = p_max.sum()
alpha_r = 5 * (p_inf / p_one)
# print(f"alpha_r = {alpha_r:.6f}  ({alpha_r*100:.2f}%)")
r_bar   = alpha_r * p_max
R = np.random.uniform(max_gen, 2*max_gen)

In [107]:
d_ref = ref.bus["PD"].values   # the “reference” nodal loads (an array of length |N|)
def sample_load(d_ref):
    gamma   = np.random.uniform(0.8, 1.2)
    sigma_ln = np.sqrt(np.log(1 + 0.05**2))
    mu_ln    = -0.5 * sigma_ln**2
    eta      = np.random.lognormal(mu_ln, sigma_ln, size=d_ref.shape)
    return gamma * eta * d_ref
# example: draw 50 000 instances
num_instance = 50000
all_d = np.stack([sample_load(d_ref) for _ in range(num_instance)], axis=0)

In [ ]:
model = gp.Model()
p = model.addVars(ref.gen_indices, vtype=gp.GRB.CONTINUOUS, name='p',
				lb=[ref.gen[g].pmin for g in ref.gen_indices],
				ub=[ref.gen[g].pmax for g in ref.gen_indices])

# p.start = [ref.gen[g].pstart for g in ref.gen_indices]
# omega = model.addVars(ref.bus_indices, vtype=gp.GRB.CONTINUOUS, name='omega')
# def busvalue(ref, i): 
# 	return gp.quicksum(p[g] for g in ref.bus[i].gens) + omega[i] - ref.bus[i].pd - ref.bus[i].gs
# def theta(ref, busvalue, i): 
# 	return gp.quicksum(ref.pi[i,j]*busvalue(ref,j) for j in ref.bus_indices)
# def lineflow(l): 
# 	return ref.line[l].one_over_reactance*(
# 	theta(ref,busvalue,ref.line[l].frombus) - theta(ref,busvalue,ref.line[l].tobus)
# 	)
# model.addConstrs((lineflow(l) <= ref.line[l].rate for l in ref.line_indices), name='c')
# model.addConstrs((lineflow(l) >= -ref.line[l].rate for l in ref.line_indices), name='c')
# model.addConstr(0 == gp.quicksum(gp.quicksum(p[g] for g in ref.bus[b].gens) +omega[b]-ref.bus[b].pd-ref.bus[b].gs for b in ref.bus_indices), name='c')
# model.setObjective(cost(ref,p), gp.GRB.MINIMIZE)